# 02. Data Cleaning and EDA

**Project:** Aura (Dormathon 2026, Track 1: Predict Early, Decide Better)

**Where we are:** notebook 01 showed that the raw file cannot be trained on. Distances were random, speeds were impossible, delivery windows made no sense, and there was no outcome column.

**Goal of this notebook:**
1. Show how the raw file is rebuilt into a clean one, and prove the rebuild fixed the problems.
2. Show the safety check that stops bad input from crashing the app.
3. Explore the clean file to see what separates failed shipments from safe ones.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pio.renderers.default = "plotly_mimetype+notebook_connected"

# Works whether the notebook is run from the project root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))  # lets us import our own code from src/

# Chart colours, kept the same in every chart
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb"


def style(fig, title, height=420):
    """Give every chart the same plain look."""
    fig.update_layout(
        title=dict(text=title, font=dict(size=16, color=INK), x=0, xanchor="left"),
        height=height,
        paper_bgcolor=SURFACE,
        plot_bgcolor=SURFACE,
        font=dict(family="Segoe UI, Arial, sans-serif", size=12, color=MUTED),
        margin=dict(l=64, r=32, t=72, b=56),
        hoverlabel=dict(bgcolor="white", font_color=INK),
    )
    fig.update_xaxes(gridcolor=GRID, linecolor=GRID, zeroline=False)
    fig.update_yaxes(gridcolor=GRID, linecolor=GRID, zeroline=False)
    return fig

## Part A. Cleaning

### 1. Rebuild the file

The cleaning lives in one script, `src/build_dataset.py`, so anyone on the team can re-create the clean file with one command: `python src/build_dataset.py`.

Here we run the same code and confirm it gives exactly the file saved in `data/processed/`. It uses a fixed random seed, so the result is the same every time.

In [2]:
from src.build_dataset import OUT_PATH, OUTCOME_COLUMNS, RAW_PATH, build

raw = pd.read_csv(RAW_PATH)
clean = pd.read_csv(OUT_PATH)
rebuilt = build(raw).reset_index(drop=True)

try:
    pd.testing.assert_frame_equal(rebuilt, clean, check_dtype=False)
    print("Rebuild matches the saved clean file exactly: True")
except AssertionError:
    print("Rebuild matches the saved clean file exactly: False. Re-run python src/build_dataset.py")

print(f"Raw file   : {raw.shape[0]:,} rows, {raw.shape[1]} columns")
print(f"Clean file : {clean.shape[0]:,} rows, {clean.shape[1]} columns")
print(f"Missing values in the clean file: {int(clean.isna().sum().sum())}")
clean.head()

Rebuild matches the saved clean file exactly: True
Raw file   : 5,000 rows, 15 columns
Clean file : 5,000 rows, 22 columns
Missing values in the clean file: 0


,Shipment_ID,Place_of_Origin,Place_of_Destination,Distance_km,Transport_Mode,Handover_Count,Lead_Time_Days,Planned_Transit_Hours,Transit_Duration_Hours,Stock_Volume,Geopolitical_Risk_Score,Fuel_Price_Index,Carrier_Reliability_Score,Buffer_of_Package,Storage_standard,Cargo_Value_USD,Hospital_Urgency_Tier,Actual_Transit_Hours,Late_Delivery,Temperature_Excursion,Handling_Damage,Shipment_Failure
0,SC-10000,Singapore Pharmaceutical Logistics Depot,Los Angeles Medical Centre Depot,14869.0,Multi-Modal Air-to-Land,5,3.12,38.0,39.6,6241,5.0,2.43,0.865,67.9,2°C to 8°C,28618.15,Routine Restock,41.1,0,0,0,0
1,SC-10001,Singapore Pharmaceutical Logistics Depot,Shanghai Medical Centre Depot,4164.0,Multi-Modal Air-to-Land,4,1.30,19.6,21.4,280,7.5,2.30,0.592,46.3,2°C to 8°C,56468.28,Elevated Procedure,27.8,0,0,0,0
2,SC-10002,Rotterdam Pharmaceutical Logistics Depot,Los Angeles Medical Centre Depot,9609.0,Multi-Modal Air-to-Land,3,2.32,29.3,33.8,279,5.6,1.78,0.673,60.3,Deep Frozen / Dry Ice,29990.67,Routine Restock,35.4,0,0,0,0
3,SC-10003,Busan Pharmaceutical Logistics Depot,Hamburg Medical Centre Depot,9126.0,Multi-Modal Air-to-Land,3,1.45,25.7,25.9,732,0.8,3.20,0.832,40.3,-20°C,49255.63,Critical Emergency,29.1,0,0,0,0
4,SC-10004,Busan Pharmaceutical Logistics Depot,Singapore Medical Centre Depot,4707.0,Air,2,0.84,11.7,12.6,3800,1.9,2.77,0.741,23.2,2°C to 8°C,42010.69,Routine Restock,13.5,0,0,0,0


### 2. What changed, column by column

For every column in the clean file: was it kept, rebuilt or added, and what share of its values is identical to the raw file.

In [3]:
what_happened = {
    "Shipment_ID": "kept",
    "Place_of_Origin": "label tidied: '(synthetic)' removed",
    "Place_of_Destination": "label tidied: '(synthetic)' removed",
    "Distance_km": "rebuilt: real distance between the two cities",
    "Transport_Mode": "kept, except European hops under 600 km switched to road",
    "Handover_Count": "added: number of times the cargo changes hands",
    "Lead_Time_Days": "rebuilt: window sized to the planned trip and the urgency tier",
    "Planned_Transit_Hours": "added: scheduled door-to-door time",
    "Transit_Duration_Hours": "rebuilt: plan plus the delay known so far (current ETA)",
    "Stock_Volume": "kept",
    "Geopolitical_Risk_Score": "kept",
    "Fuel_Price_Index": "kept",
    "Carrier_Reliability_Score": "kept",
    "Buffer_of_Package": "rebuilt: hold time from dispatch, packaging chosen for the trip",
    "Storage_standard": "kept",
    "Cargo_Value_USD": "kept",
    "Hospital_Urgency_Tier": "label tidied: long suffix removed",
    "Actual_Transit_Hours": "added: simulated outcome (known only after delivery)",
    "Late_Delivery": "added: simulated outcome (known only after delivery)",
    "Temperature_Excursion": "added: simulated outcome (known only after delivery)",
    "Handling_Damage": "added: simulated outcome (known only after delivery)",
    "Shipment_Failure": "added: the target, 1 if any of the three failures happened",
}

pd.set_option("display.max_colwidth", 120)  # show the full sentence
changes = pd.DataFrame({"what_happened": what_happened})
changes["values_same_as_raw"] = [
    f"{(raw[col] == clean[col]).mean():.0%}" if col in raw.columns else "new column"
    for col in changes.index
]
changes

,what_happened,values_same_as_raw
Shipment_ID,kept,100%
Place_of_Origin,label tidied: '(synthetic)' removed,0%
Place_of_Destination,label tidied: '(synthetic)' removed,0%
Distance_km,rebuilt: real distance between the two cities,0%
Transport_Mode,"kept, except European hops under 600 km switched to road",95%
Handover_Count,added: number of times the cargo changes hands,new column
Lead_Time_Days,rebuilt: window sized to the planned trip and the urgency tier,0%
Planned_Transit_Hours,added: scheduled door-to-door time,new column
Transit_Duration_Hours,rebuilt: plan plus the delay known so far (current ETA),0%
Stock_Volume,kept,100%


### 3. Did the rebuild fix the problems from notebook 01?

We run the same physical checks on both files and put the results side by side.

One difference in how we measure: the raw file has only one travel-time column. The clean file separates the **planned** time from the **current ETA**, so speed and window checks use the planned time.

In [4]:
JET_KMH, TRUCK_KMH = 900, 90


def city(place):
    """'Los Angeles Medical Centre Depot (synthetic)' -> 'Los Angeles'"""
    return place.str.replace(r"\s+(Pharmaceutical Logistics|Medical Centre) Depot.*$", "", regex=True)


def physics_checks(df, trip_col):
    """Count the rows that break each rule. trip_col is the planned travel time in hours."""
    routes = (
        df.assign(o=city(df["Place_of_Origin"]), d=city(df["Place_of_Destination"]))
        .groupby(["o", "d"])["Distance_km"].agg(["min", "max"])
    )
    speed = df["Distance_km"] / df[trip_col]
    is_land = df["Transport_Mode"] == "Land"
    window_hours = df["Lead_Time_Days"] * 24
    return pd.Series({
        "Routes whose longest distance is over 1.5x the shortest": int((routes["max"] / routes["min"] > 1.5).sum()),
        "Air or multi-modal shipments faster than a jet": int((~is_land & (speed > JET_KMH)).sum()),
        "Land shipments faster than a truck": int((is_land & (speed > TRUCK_KMH)).sum()),
        "Delivery window shorter than the planned trip": int((window_hours < df[trip_col]).sum()),
        "Delivery window longer than 30 days": int((df["Lead_Time_Days"] > 30).sum()),
        "Has an outcome column to predict": "Shipment_Failure" in df.columns,
    })


before_after = pd.DataFrame({
    "raw_file": physics_checks(raw, "Transit_Duration_Hours"),
    "clean_file": physics_checks(clean, "Planned_Transit_Hours"),
})
before_after

,raw_file,clean_file
Routes whose longest distance is over 1.5x the shortest,64,0
Air or multi-modal shipments faster than a jet,631,0
Land shipments faster than a truck,119,0
Delivery window shorter than the planned trip,516,0
Delivery window longer than 30 days,877,0
Has an outcome column to predict,False,True


In [5]:
mode_colors = {"Air": BLUE, "Multi-Modal Air-to-Land": ORANGE, "Land": AQUA}

fig = make_subplots(rows=1, cols=2, shared_yaxes=False, horizontal_spacing=0.09,
                    subplot_titles=["Raw file", "Clean file"])
for col, (df, trip_col) in enumerate([(raw, "Transit_Duration_Hours"), (clean, "Planned_Transit_Hours")], start=1):
    for mode, color in mode_colors.items():
        part = df[df["Transport_Mode"] == mode]
        fig.add_trace(go.Scatter(
            x=part["Distance_km"], y=part[trip_col], mode="markers", name=mode,
            legendgroup=mode, showlegend=col == 1,
            marker=dict(color=color, size=4, opacity=0.5),
            hovertemplate=mode + "<br>%{x:,.0f} km<br>%{y:.1f} h<extra></extra>",
        ), row=1, col=col)
    fig.update_xaxes(title_text="Distance (km)", row=1, col=col)
fig.update_yaxes(title_text="Travel time (hours)", row=1, col=1)
fig.update_annotations(font=dict(size=12, color=INK))
style(fig, "Travel time against distance, before and after cleaning", height=480)
fig.update_layout(legend=dict(orientation="h", y=-0.22, x=0, font=dict(color=INK)), margin=dict(b=96))
fig.show()

**What this tells us:**

- Every physical problem counted in notebook 01 is now at zero.
- In the raw chart the dots are a cloud: distance tells you nothing about travel time. In the clean chart each transport mode forms its own line: longer routes take longer, trucks are slower than aircraft, and long-haul flights show a step where a stopover is added.
- The clean file has an outcome column, so there is finally something to predict.

### 4. The safety check for new input

A coordinator will not always give the app tidy data. `src/preprocessing.py` has one function, `clean_raw_shipment_data`, that every shipment passes through before it reaches the model. It does three jobs:

1. Accepts different spellings of the same column and category.
2. Checks that every number is in a sensible range.
3. Refuses bad rows with a clear message instead of crashing or guessing.

First, the clean file itself should pass.

In [6]:
from src.preprocessing import ShipmentValidationError, clean_raw_shipment_data

validated = clean_raw_shipment_data(clean, require_target=True)
print(f"Clean file passed the safety check: {len(validated):,} of {len(clean):,} rows accepted")

Clean file passed the safety check: 5,000 of 5,000 rows accepted


Next, messy but understandable input. Different column names and different spellings should be accepted and standardised.

In [7]:
messy = clean.head(3).drop(columns=OUTCOME_COLUMNS).copy()
messy["Transport_Mode"] = ["road", "AIR FREIGHT", "multimodal"]
messy["Storage_standard"] = ["2-8 C", "dry ice", "minus 20 C"]
messy["Hospital_Urgency_Tier"] = ["routine", "Critical", "elevated"]
messy = messy.rename(columns={"Carrier_Reliability_Score": "carrier reliability", "Buffer_of_Package": "buffer hours"})

tidied = clean_raw_shipment_data(messy)
pd.DataFrame({
    "typed_mode": messy["Transport_Mode"], "read_as_mode": tidied["Transport_Mode"],
    "typed_storage": messy["Storage_standard"], "read_as_storage": tidied["Storage_standard"],
    "typed_urgency": messy["Hospital_Urgency_Tier"], "read_as_urgency": tidied["Hospital_Urgency_Tier"],
})

,typed_mode,read_as_mode,typed_storage,read_as_storage,typed_urgency,read_as_urgency
0,road,Land,2-8 C,2°C to 8°C,routine,Routine Restock
1,AIR FREIGHT,Air,dry ice,Deep Frozen / Dry Ice,Critical,Critical Emergency
2,multimodal,Multi-Modal Air-to-Land,minus 20 C,-20°C,elevated,Elevated Procedure


Last, input that is actually wrong: a reliability score above 1, a transport mode that does not exist, and a missing buffer value.

In [8]:
broken = clean.head(5).drop(columns=OUTCOME_COLUMNS).copy()
broken.loc[0, "Carrier_Reliability_Score"] = 1.7      # must be between 0 and 1
broken.loc[1, "Transport_Mode"] = "Ship"              # not a mode we know
broken.loc[2, "Buffer_of_Package"] = np.nan           # missing

try:
    clean_raw_shipment_data(broken)
    print("Accepted (this should not happen)")
except ShipmentValidationError as problem:
    print("Refused with this message:\n")
    print(str(problem).replace("; ", ";\n").replace(". ", ".\n"))

Refused with this message:

Shipment validation failed:
- Carrier_Reliability_Score: expected a finite score from 0 to 1;
invalid data rows 1
- Buffer_of_Package: expected a finite number greater than or equal to 0;
invalid data rows 3
- Transport_Mode: choose one of ['Air', 'Land', 'Multi-Modal Air-to-Land'];
invalid data rows 2


**What this tells us:** the app never scores a shipment it cannot trust. It names the column and the row that is wrong, so the coordinator can fix it. This is what sits behind the on-screen message "Insufficient data for reliable prediction".

## Part B. Exploring the clean file

From here on we only use the clean file. The question for every chart is the same: **what is different about the shipments that fail?**

### 5. How often do shipments fail, and how?

A shipment counts as failed if any of three things happens: it arrives after the delivery window, its temperature goes out of range, or it is damaged in handling.

In [9]:
outcomes = {
    "Any failure (the target)": "Shipment_Failure",
    "Late delivery": "Late_Delivery",
    "Temperature excursion": "Temperature_Excursion",
    "Handling damage": "Handling_Damage",
}
rates = pd.Series({label: clean[col].mean() for label, col in outcomes.items()}).sort_values()
counts = pd.Series({label: int(clean[col].sum()) for label, col in outcomes.items()})[rates.index]

fig = go.Figure(go.Bar(
    x=rates.values, y=rates.index, orientation="h", marker_color=BLUE,
    text=[f"{r:.1%}  ({c:,} shipments)" for r, c in zip(rates, counts)],
    textposition="outside", textfont=dict(color=INK), cliponaxis=False,
    hovertemplate="%{y}: %{x:.1%}<extra></extra>",
))
fig.update_xaxes(tickformat=".0%", range=[0, rates.max() * 1.45], title_text="Share of all shipments")
style(fig, "How often each kind of failure happens", height=340)
fig.update_layout(bargap=0.35, margin=dict(l=200))
fig.show()

failed = clean[clean["Shipment_Failure"] == 1]
n_types = failed[["Late_Delivery", "Temperature_Excursion", "Handling_Damage"]].sum(axis=1)
print(f"Failed shipments: {len(failed):,} of {len(clean):,} ({len(failed) / len(clean):.1%})")
print(f"Failed in more than one way at once: {(n_types > 1).sum():,} ({(n_types > 1).mean():.0%} of failures)")

Failed shipments: 787 of 5,000 (15.7%)
Failed in more than one way at once: 197 (25% of failures)


**What this tells us:**

- About 1 shipment in 6 fails. The other 5 are fine. This imbalance matters later: a model that always says "safe" would be right 84% of the time and be useless, so we will judge the model on precision and recall, not on accuracy.
- Late delivery is the most common failure, then temperature excursion. Handling damage is rare.
- About a quarter of failed shipments fail in more than one way, usually late and warm together. A delay causes both.

### 6. Which kinds of shipment fail more?

In [10]:
overall = clean["Shipment_Failure"].mean()
category_cols = ["Hospital_Urgency_Tier", "Transport_Mode", "Storage_standard"]

fig = make_subplots(rows=3, cols=1, subplot_titles=category_cols, vertical_spacing=0.16)
for i, col in enumerate(category_cols, start=1):
    grouped = clean.groupby(col)["Shipment_Failure"].agg(["mean", "size"]).sort_values("mean")
    fig.add_trace(go.Bar(
        x=grouped["mean"], y=grouped.index, orientation="h", marker_color=BLUE,
        text=[f"{m:.1%}  (of {n:,})" for m, n in zip(grouped["mean"], grouped["size"])],
        textposition="outside", textfont=dict(color=INK), cliponaxis=False,
        hovertemplate="%{y}: %{x:.1%}<extra></extra>",
    ), row=i, col=1)
    fig.add_vline(x=overall, line=dict(color=MUTED, width=1, dash="dash"), layer="below", row=i, col=1)
fig.update_annotations(font=dict(size=12, color=INK), x=0, xanchor="left")
fig.add_annotation(xref="x", yref="paper", x=overall, y=1.04, text=f"All shipments: {overall:.1%}",
                   showarrow=False, font=dict(color=INK), xanchor="left")
fig.update_xaxes(tickformat=".0%", range=[0, 0.42])
style(fig, "Failure rate by category", height=620)
fig.update_layout(showlegend=False, bargap=0.35, margin=dict(l=200))
fig.show()

**What this tells us:**

- **Urgency is the biggest gap.** Critical Emergency shipments fail about two and a half times as often as Routine Restock. They are given a tighter delivery window, so the same delay is more likely to make them late. These are also the shipments a hospital can least afford to lose.
- Air-only shipments fail least. Multi-modal and land shipments have more handovers and more time on the road.
- Frozen cargo fails a little more often than chilled cargo, because its packaging loses hold time faster.

### 7. Which numbers move with failure?

We split four inputs into bands and look at the failure rate inside each band. Two are given in the file. Two are simple differences we compute here:

- **Buffer beyond ETA** = `Buffer_of_Package` minus `Transit_Duration_Hours`. How many hours of cold are left over if the shipment arrives when currently expected.
- **Slack before the window closes** = `Lead_Time_Days` x 24 minus `Transit_Duration_Hours`. How many hours the shipment can still lose before it is late.

In [11]:
view = clean.copy()
view["buffer_beyond_eta"] = view["Buffer_of_Package"] - view["Transit_Duration_Hours"]
view["slack_before_window"] = view["Lead_Time_Days"] * 24 - view["Transit_Duration_Hours"]

panels = [
    ("Carrier_Reliability_Score", [0.49, 0.6, 0.7, 0.8, 0.9, 1.0], ["0.5 to 0.6", "0.6 to 0.7", "0.7 to 0.8", "0.8 to 0.9", "0.9 to 1.0"], "Carrier reliability score"),
    ("Geopolitical_Risk_Score", [-0.1, 2, 4, 6, 8, 10], ["0 to 2", "2 to 4", "4 to 6", "6 to 8", "8 to 10"], "Geopolitical risk score"),
    ("buffer_beyond_eta", [-999, 0, 6, 12, 24, 999], ["0 or less", "0 to 6", "6 to 12", "12 to 24", "Over 24"], "Buffer beyond ETA (hours)"),
    ("slack_before_window", [-999, 0, 3, 6, 12, 999], ["0 or less", "0 to 3", "3 to 6", "6 to 12", "Over 12"], "Slack before the window closes (hours)"),
]

fig = make_subplots(rows=2, cols=2, subplot_titles=[p[3] for p in panels],
                    vertical_spacing=0.2, horizontal_spacing=0.1)
band_tables = {}
for i, (col, edges, labels, title) in enumerate(panels):
    bands = pd.cut(view[col], bins=edges, labels=labels)
    grouped = view.groupby(bands, observed=False)["Shipment_Failure"].agg(["mean", "size"])
    band_tables[title] = grouped
    fig.add_trace(go.Bar(
        x=grouped.index.astype(str), y=grouped["mean"], marker_color=BLUE,
        text=[f"{m:.0%}" for m in grouped["mean"]], textposition="outside",
        textfont=dict(color=INK), cliponaxis=False, customdata=grouped["size"],
        hovertemplate="%{x}<br>Failure rate: %{y:.1%}<br>Shipments: %{customdata:,}<extra></extra>",
    ), row=i // 2 + 1, col=i % 2 + 1)
    fig.add_hline(y=overall, line=dict(color=MUTED, width=1, dash="dash"), layer="below", row=i // 2 + 1, col=i % 2 + 1)
fig.update_annotations(font=dict(size=12, color=INK))
fig.update_yaxes(tickformat=".0%", range=[0, 1.15])
fig.update_yaxes(title_text="Failure rate", col=1)
style(fig, "Failure rate by band (dashed line = all shipments)", height=640)
fig.update_layout(showlegend=False, bargap=0.3)
fig.show()

pd.concat(band_tables).rename(columns={"mean": "failure_rate", "size": "shipments"}).round(3)

failure_rate  shipments
Carrier reliability score              0.5 to 0.6         0.254        957
                                       0.6 to 0.7         0.189        976
                                       0.7 to 0.8         0.146        999
                                       0.8 to 0.9         0.111       1047
                                       0.9 to 1.0         0.096       1021
Geopolitical risk score                0 to 2             0.122        987
                                       2 to 4             0.140        968
                                       4 to 6             0.143       1001
                                       6 to 8             0.194       1062
                                       8 to 10            0.185        982
Buffer beyond ETA (hours)              0 or less          1.000        125
                                       0 to 6             0.581        160
                                       6 to 12            0.203        403
                                       12 to 24           0.124       2194
                                       Over 24            0.102       2118
Slack before the window closes (hours) 0 or less          1.000        104
                                       0 to 3             0.424        467
                                       3 to 6             0.145        812
                                       6 to 12            0.113       1759
                                       Over 12            0.091       1858

**What this tells us:**

- **Carrier reliability** matters steadily: the least reliable carriers fail about two and a half times as often as the most reliable ones.
- **Geopolitical risk** matters a little. Failure rises on the riskiest routes.
- **Buffer beyond ETA** is the sharpest signal in the file. With more than 12 hours to spare, a shipment is close to the average risk. With under 6 hours to spare, more than half fail. At zero or below, all fail.
- **Slack before the window closes** behaves the same way for late delivery: under 3 hours of slack, about 4 in 10 fail.

The two computed columns are far more telling than the columns they are made from. That is the starting point for notebook 03.

### 8. Failures that are already certain

Some shipments are doomed by plain arithmetic: the current ETA already uses up the whole delivery window, or the whole packaging buffer.

In [12]:
eta_past_window = view["slack_before_window"] <= 0   # no slack left
buffer_too_short = view["buffer_beyond_eta"] <= 0     # no buffer left on arrival
certain = eta_past_window | buffer_too_short

print(f"ETA at or past the delivery window   : {eta_past_window.sum():,} shipments, {view.loc[eta_past_window, 'Shipment_Failure'].mean():.0%} fail")
print(f"ETA as long as the buffer or longer  : {buffer_too_short.sum():,} shipments, {view.loc[buffer_too_short, 'Shipment_Failure'].mean():.0%} fail")
print(f"Either of the two                    : {certain.sum():,} shipments")
print()
print(f"Share of all failures that are certain in this way: {certain.sum() / view['Shipment_Failure'].sum():.0%}")
print(f"Failure rate among the remaining shipments         : {view.loc[~certain, 'Shipment_Failure'].mean():.1%}")

ETA at or past the delivery window   : 104 shipments, 100% fail
ETA as long as the buffer or longer  : 125 shipments, 100% fail
Either of the two                    : 217 shipments

Share of all failures that are certain in this way: 28%
Failure rate among the remaining shipments         : 11.9%


**What this tells us:** about a quarter of failures can be spotted with a calculator. They still belong at the top of the coordinator's queue, but they do not prove the model is clever. The harder and more useful job is the other three quarters, where nothing looks wrong yet. When we score the model in notebooks 04 and 05, we should report how it does on those shipments as well as overall.

### 9. How the columns relate to each other now

In notebook 01 the strongest link between any two raw columns was 0.34. Here is the same picture for the clean inputs, with the target added as the last row.

In [13]:
input_numeric = [
    "Distance_km", "Planned_Transit_Hours", "Transit_Duration_Hours", "Lead_Time_Days",
    "Buffer_of_Package", "Handover_Count", "Carrier_Reliability_Score",
    "Geopolitical_Risk_Score", "Fuel_Price_Index", "Stock_Volume", "Cargo_Value_USD",
]
corr = clean[input_numeric + ["Shipment_Failure"]].corr().round(2)

fig = go.Figure(go.Heatmap(
    z=corr.values, x=corr.columns, y=corr.index, zmin=-1, zmax=1,
    colorscale=[[0, "#2a78d6"], [0.5, "#f0efec"], [1, "#e34948"]],
    text=corr.values, texttemplate="%{text:.2f}", textfont=dict(color=INK, size=10),
    xgap=2, ygap=2, colorbar=dict(title="Correlation", thickness=12),
    hovertemplate="%{y}<br>%{x}<br>Correlation: %{z:.2f}<extra></extra>",
))
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(tickangle=-35)
style(fig, "Correlation between clean inputs and the target", height=660)
fig.update_layout(margin=dict(l=190, b=150))
fig.show()

print("Link between each input and Shipment_Failure, strongest first:")
corr["Shipment_Failure"].drop("Shipment_Failure").sort_values(key=abs, ascending=False).to_frame("correlation")

Link between each input and Shipment_Failure, strongest first:


,correlation
Buffer_of_Package,-0.17
Carrier_Reliability_Score,-0.16
Lead_Time_Days,-0.09
Geopolitical_Risk_Score,0.07
Transit_Duration_Hours,0.05
Distance_km,-0.04
Planned_Transit_Hours,-0.03
Handover_Count,0.02
Stock_Volume,0.01
Fuel_Price_Index,0.00


**What this tells us:**

- Distance, planned time, current ETA, delivery window and buffer now move together, as they do for real shipments.
- No single raw input is strongly linked to failure on its own. Failure depends on **gaps between columns** (buffer against ETA, window against ETA), which a plain correlation cannot see. That is why we build those gaps as features in notebook 03.
- `Fuel_Price_Index`, `Stock_Volume` and `Cargo_Value_USD` are not linked to failure. Cargo value is still important, not for predicting failure but for deciding which shipment to save first.

### 10. What a failure costs

The prediction only matters because of what is lost when a shipment fails.

In [14]:
value_all = clean["Cargo_Value_USD"].sum()
value_failed = failed["Cargo_Value_USD"].sum()
print(f"Cargo value across all simulated shipments : ${value_all:,.0f}")
print(f"Cargo value in shipments that failed       : ${value_failed:,.0f} ({value_failed / value_all:.1%})")
print(f"Average value of one failed shipment       : ${failed['Cargo_Value_USD'].mean():,.0f}")

by_tier = clean.groupby("Hospital_Urgency_Tier").agg(
    shipments=("Shipment_ID", "size"),
    failed=("Shipment_Failure", "sum"),
    failure_rate=("Shipment_Failure", "mean"),
)
by_tier["value_in_failed_usd"] = failed.groupby("Hospital_Urgency_Tier")["Cargo_Value_USD"].sum()
by_tier.sort_values("failure_rate", ascending=False).style.format(
    {"failure_rate": "{:.1%}", "value_in_failed_usd": "${:,.0f}", "shipments": "{:,}", "failed": "{:,}"}
)

Cargo value across all simulated shipments : $352,381,450
Cargo value in shipments that failed       : $55,384,981 (15.7%)
Average value of one failed shipment       : $70,375


,shipments,failed,failure_rate,value_in_failed_usd
Hospital_Urgency_Tier,,,,
Critical Emergency,757,223,29.5%,"$15,684,600"
Elevated Procedure,"1,547",258,16.7%,"$17,974,370"
Routine Restock,"2,696",306,11.4%,"$21,726,012"


**What this tells us:** the failed shipments carry tens of millions of dollars of simulated cargo, and the most urgent tier loses the largest share of its shipments. Counting the full cargo value as lost is an upper bound, since a late shipment is not always written off. The app should state that assumption wherever it shows money saved.

### 11. Columns the model must never see

Four columns describe what happened **after** delivery. A coordinator watching a live shipment does not have them. If the model trained on them it would look perfect in testing and be useless in use. This is called leakage.

In [15]:
LEAKAGE_COLUMNS = ["Actual_Transit_Hours", "Late_Delivery", "Temperature_Excursion", "Handling_Damage"]
TARGET = "Shipment_Failure"
SAFE_INPUTS = [c for c in clean.columns if c not in LEAKAGE_COLUMNS + [TARGET]]

assert set(LEAKAGE_COLUMNS + [TARGET]) == set(OUTCOME_COLUMNS)
print("Never use as model inputs:", LEAKAGE_COLUMNS)
print("Target:", TARGET)
print(f"Safe to use ({len(SAFE_INPUTS)}):", SAFE_INPUTS)

Never use as model inputs: ['Actual_Transit_Hours', 'Late_Delivery', 'Temperature_Excursion', 'Handling_Damage']
Target: Shipment_Failure
Safe to use (17): ['Shipment_ID', 'Place_of_Origin', 'Place_of_Destination', 'Distance_km', 'Transport_Mode', 'Handover_Count', 'Lead_Time_Days', 'Planned_Transit_Hours', 'Transit_Duration_Hours', 'Stock_Volume', 'Geopolitical_Risk_Score', 'Fuel_Price_Index', 'Carrier_Reliability_Score', 'Buffer_of_Package', 'Storage_standard', 'Cargo_Value_USD', 'Hospital_Urgency_Tier']


## What we found and what happens next

### Findings

| # | Finding | What we do about it |
|---|---|---|
| 1 | The rebuild is repeatable and fixes every physical problem from notebook 01 | Train on `data/processed/aura_shipments.csv` |
| 2 | The safety check accepts messy spellings and refuses wrong values with a clear message | Every shipment goes through it before scoring |
| 3 | About 16% of shipments fail: late delivery most often, then temperature excursion, then handling damage | Judge the model on precision and recall, not accuracy |
| 4 | Critical Emergency shipments fail about two and a half times as often as Routine Restock | Urgency belongs in the triage ranking |
| 5 | Buffer beyond ETA and slack before the window closes are the sharpest signals | Build them as features in notebook 03 |
| 6 | About a quarter of failures are certain from arithmetic alone | Report model results with and without these shipments |
| 7 | Cargo value does not predict failure | Use it to rank and to show money saved, not to predict |
| 8 | Four columns are only known after delivery | Never give them to the model |

### Next

Notebook 03 turns these findings into model features and tests how much each one helps.

### Limits we state openly

- Outcomes are simulated from rules we wrote, so the patterns above are the ones we built in, plus random noise. They show the method works. They do not show how often real shipments fail.
- Money figures count the full cargo value as lost when a shipment fails.